# Testing pose estimation

In [ ]:
import torch

print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

import cv2
import imageio.v3 as iio
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
from aitraf.utils import get_video_rotation_deg

CLIPS_DIR = Path("../data/clips")
YOLO_WEIGHTS = Path("../models/yolo11n-pose.pt")

CLIP_INDEX = 3
FRAME_INDEX = 100

#### Load video

In [ ]:
clip_path = sorted(CLIPS_DIR.iterdir())[CLIP_INDEX]

clip_path

#### Extract a frame

In [ ]:
# Load selected frame from the local video
frame_rgb = iio.imread(clip_path, index=FRAME_INDEX)
frame_rgb = np.rot90(frame_rgb, k=get_video_rotation_deg(clip_path) // 90)
frame_bgr = cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR)
print(f"Loaded frame {FRAME_INDEX} from {clip_path.name}")

In [ ]:
iio.immeta(str(clip_path), plugin="pyav")

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(frame_rgb)
plt.title(f"Frame: {FRAME_INDEX}")
plt.axis("off")
plt.show()

#### Test on a frame

In [ ]:
model = YOLO(YOLO_WEIGHTS)

In [ ]:
frame_pose = model.predict(
    source=frame_bgr, device="cuda", imgsz=640, conf=0.5, verbose=False, max_det=1
)

vis = frame_pose[0].plot()
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

#### Test on full video

In [ ]:
results = model.predict(
    source=clip_path.as_posix(),
    device="cuda",
    imgsz=640,
    conf=0.5,
    save=True,
    verbose=False,
    max_det=1,
)